*Part 2 of 4 · Algorithms* · [← index](README.md)

# Algorithms

---
## 1. Binary search

**Complexity:** O(log n) time, O(1) space. O(n log m) when searching the answer space.

Two shapes, and mixing them is an infinite loop. Exact match: `while l <= r`, `r` starts at
`len - 1`, move with `r = m - 1`, return from inside the loop. Lower bound: `while l < r`,
`r` starts at `len`, move with `r = m`, return `l` after the loop. Use
`m = l + (r - l) // 2` out of habit — identical in Python, overflow-safe elsewhere.

In [ ]:
def binary_search(nums, target):
    """Index of target in a sorted array, or -1."""
    # O(log n) time, O(1) space.
    l, r = 0, len(nums) - 1
    while l <= r:                 # <= : a single-element range is still searchable
        m = l + (r - l) // 2
        if nums[m] == target:
            return m
        elif nums[m] < target:
            l = m + 1             # m is ruled out, so +1
        else:
            r = m - 1
    return -1

In [ ]:
def lower_bound(nums, target):
    """First index where nums[i] >= target. == len(nums) if target exceeds everything."""
    # O(log n) time, O(1) space.
    l, r = 0, len(nums)           # r is EXCLUSIVE here
    while l < r:                  # strict: stop when the range is empty
        m = l + (r - l) // 2
        if nums[m] < target:
            l = m + 1
        else:
            r = m                 # m might BE the answer, so keep it
    return l                      # never returns from inside the loop

In [ ]:
import math

def min_eating_speed(piles, hours):
    """Search the ANSWER space, not the array. Feasibility must be monotonic."""
    # O(n log m) time, O(1) space, m = max(piles).

    def feasible(k):              # write this helper separately, always
        return sum(math.ceil(p / k) for p in piles) <= hours

    l, r = 1, max(piles)          # slowest useful speed .. fastest ever needed
    while l < r:
        m = l + (r - l) // 2
        if feasible(m):
            r = m                 # m works, but something smaller might too
        else:
            l = m + 1
    return l

In [ ]:
def search_matrix(matrix, target):
    """Fully sorted matrix: treat it as one flat array of length rows * cols."""
    # O(log mn) time, O(1) space.
    if not matrix or not matrix[0]:
        return False
    rows, cols = len(matrix), len(matrix[0])
    l, r = 0, rows * cols - 1
    while l <= r:
        m = l + (r - l) // 2
        row, col = divmod(m, cols)            # flat index -> (row, col)
        val = matrix[row][col]
        if val == target:
            return True
        elif val < target:
            l = m + 1
        else:
            r = m - 1
    return False

---
## 2. Tree DFS

**Complexity:** O(n) time, O(h) space for the call stack — O(n) on a degenerate tree.

The three orders differ *only* in where you touch the node relative to the two recursive
calls. Inorder on a BST comes out sorted, which alone solves a surprising number of
problems. Going iterative: push right before left, because the stack pops in reverse.

In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals):
    """LeetCode level-order format, None for missing. [1,2,3,None,4] -> root."""
    # O(n) time, O(n) space.
    if not vals:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)   # only real nodes get enqueued
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

In [ ]:
def preorder(node):
    """Node first. Use when a parent must be processed before its children (copying)."""
    # O(n) time, O(h) space.
    if not node:
        return []
    return [node.val] + preorder(node.left) + preorder(node.right)


def inorder(node):
    """Node in the middle. On a BST this comes out SORTED."""
    # O(n) time, O(h) space.
    if not node:
        return []
    return inorder(node.left) + [node.val] + inorder(node.right)


def postorder(node):
    """Node last. Use when children must resolve first (deleting, computing heights)."""
    # O(n) time, O(h) space.
    if not node:
        return []
    return postorder(node.left) + postorder(node.right) + [node.val]

In [ ]:
def preorder_iterative(root):
    # O(n) time, O(h) space.
    if not root:
        return []
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right:
            stack.append(node.right)   # right pushed FIRST
        if node.left:
            stack.append(node.left)    # so left pops first
    return out

In [ ]:
def inorder_iterative(root):
    """Dive left remembering the path, then pop-visit-and-go-right."""
    # O(n) time, O(h) space.
    out, stack, curr = [], [], root
    while curr or stack:
        while curr:
            stack.append(curr)
            curr = curr.left
        curr = stack.pop()             # leftmost unvisited node
        out.append(curr.val)
        curr = curr.right
    return out

---
## 3. Tree BFS

**Complexity:** O(n) time, O(w) space where w is the widest level.

Snapshot `len(queue)` into a variable before the inner loop. The queue grows as you append
children, so reading its length mid-loop merges levels together. Anything phrased
per-level — level averages, right-side view, minimum depth — is this.

In [ ]:
def level_order(root):
    """List of levels, top to bottom."""
    # O(n) time, O(w) space.
    if not root:
        return []
    out = []
    queue = deque([root])
    while queue:
        level_size = len(queue)        # SNAPSHOT before appending children
        level = []
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        out.append(level)
    return out

---
## 4. Depth and the bottom-up return

**Complexity:** O(n) time, O(h) space.

The most reusable tree shape. The base case returns the identity value, then each node
combines its children's results and adds its own contribution. Recurse once per child and
store the result — calling it twice turns O(n) into O(2ⁿ).

In [ ]:
def max_depth(node):
    # O(n) time, O(h) space.
    if not node:
        return 0
    left = max_depth(node.left)        # store, don't recompute
    right = max_depth(node.right)
    return 1 + max(left, right)        # combine children, add self

In [ ]:
def is_balanced(node):
    """Same shape, richer return value: (balanced?, height)."""
    # O(n) time, O(h) space.
    def dfs(n):
        if not n:
            return True, 0
        lb, lh = dfs(n.left)
        rb, rh = dfs(n.right)
        return (lb and rb and abs(lh - rh) <= 1), 1 + max(lh, rh)
    return dfs(node)[0]